# LLM Zoomcamp 2026 — Homework 4: Evaluation

This notebook follows the official **Homework: Evaluation** instructions.

It evaluates text, vector, and hybrid search over the 72 course lesson pages pinned to commit `8c1834d`.

> Run the notebook from the Homework 4 project directory. The project must reuse `embedder.py`, the downloaded ONNX model, and the search setup from Homework 2.

## 0. Required project files

Before running the notebook, verify that the project contains:

- `embedder.py`
- `download.py`
- `models/Xenova/all-MiniLM-L6-v2/model.onnx`
- `evaluation_utils.py`
- `rag_helper.py`
- `ground-truth.csv`

Install the libraries specified in the homework:

```bash
uv add openai pydantic python-dotenv pandas
```

The Homework 2 environment also needs `onnxruntime`, `tokenizers`, `numpy`, `tqdm`, `minsearch`, and `gitsource`.

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

## 1. Load the 72 lesson pages

The repository is pinned to commit `8c1834d`, as required by the homework.

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of lesson pages:", len(documents))
assert len(documents) == 72

## 2. Question 1 — Generate questions for the first three pages

For each of the first three Agentic RAG lesson pages, the LLM generates five student questions. The answer is the average number of input tokens across the three calls.

The available choices are `140`, `1400`, `14000`, and `140000`.

In [ ]:
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

from evaluation_utils import llm_structured

load_dotenv()
client = OpenAI()


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
first_three_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

first_three_documents = [
    document
    for document in documents
    if document["filename"] in first_three_filenames
]

first_three_documents = sorted(
    first_three_documents,
    key=lambda document: first_three_filenames.index(document["filename"]),
)

assert len(first_three_documents) == 3
[document["filename"] for document in first_three_documents]

In [ ]:
generated_questions = []
usages = []

for document in first_three_documents:
    user_prompt = json.dumps(
        {
            "filename": document["filename"],
            "content": document["content"],
        }
    )

    questions, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gpt-5.4-mini",
    )

    generated_questions.append(
        {
            "filename": document["filename"],
            "questions": questions.questions,
        }
    )
    usages.append(usage)

input_tokens = [usage.input_tokens for usage in usages]
average_input_tokens = sum(input_tokens) / len(input_tokens)

print("Input tokens per call:", input_tokens)
print("Average input tokens:", average_input_tokens)
print("Q1: Select the closest option among 140, 1400, 14000, and 140000.")

## 3. Load the provided full ground truth

The official file contains 360 questions. Each record has a `question` and the `filename` of the lesson page that should answer it.

In [ ]:
import pandas as pd

ground_truth_path = Path("ground-truth.csv")
if not ground_truth_path.exists():
    alternative_path = Path("../data/ground_truth.csv")
    if alternative_path.exists():
        ground_truth_path = alternative_path

ground_truth_df = pd.read_csv(ground_truth_path)
ground_truth = ground_truth_df.to_dict(orient="records")

print("Ground-truth file:", ground_truth_path)
print("Number of questions:", len(ground_truth))
ground_truth_df.head()

## 4. Create the same chunks as in Homework 2

The required configuration is `size=2000` and `step=1000`. It should produce 295 chunks.

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Number of chunks:", len(chunks))
assert len(chunks) == 295

## 5. Rebuild text and vector search from Homework 2

Both indexes are keyed on `filename`. The text index searches the chunk `content`; the vector index uses the ONNX `Embedder` from Homework 2.

In [ ]:
import numpy as np
from minsearch import Index, VectorSearch
from embedder import Embedder

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)


def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results,
    )

In [ ]:
embedder = Embedder()

embeddings = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(embeddings, chunks)


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(
        query_vector,
        num_results=num_results,
    )

## 6. Question 2 — First result from text search

In [ ]:
q = ground_truth[0]["question"]
text_results = text_search(q)
q2_answer = text_results[0]["filename"]

print("First ground-truth question:", q)
print("Q2 answer:", q2_answer)

## 7. Question 3 — First result from vector search

In [ ]:
vector_results = vector_search(q)
q3_answer = vector_results[0]["filename"]

print("Q3 answer:", q3_answer)

## 8. Evaluation metrics

A returned chunk is relevant when its `filename` matches the ground-truth `filename`. Hit Rate checks whether the correct page appears anywhere in the returned list. MRR also rewards a higher rank.

In [ ]:
def compute_relevance(search_function, ground_truth):
    relevance_total = []

    for record in ground_truth:
        results = search_function(record["question"])
        relevance = [
            result["filename"] == record["filename"]
            for result in results
        ]
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] is True:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)


def evaluate(search_function, ground_truth):
    relevance_total = compute_relevance(search_function, ground_truth)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## 9. Question 4 — Evaluate text search

In [ ]:
text_search_metrics = evaluate(text_search, ground_truth)
q4_value = text_search_metrics["hit_rate"]

print("Text search metrics:", text_search_metrics)
print("Q4 Hit Rate:", q4_value)
print("Select the closest option among 0.55, 0.66, 0.76, and 0.88.")

## 10. Question 5 — Evaluate vector search

In [ ]:
vector_search_metrics = evaluate(vector_search, ground_truth)
q5_value = vector_search_metrics["mrr"]

print("Vector search metrics:", vector_search_metrics)
print("Q5 MRR:", q5_value)
print("Select the closest option among 0.35, 0.45, 0.55, and 0.65.")

## 11. Question 6 — Tune hybrid search

The RRF implementation and `hybrid_search` definition below are the ones provided in the homework.

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
hybrid_results = {}

for k in [1, 50, 100, 200]:
    search_function = lambda query, k=k: hybrid_search(query, k=k)
    hybrid_results[k] = evaluate(search_function, ground_truth)

hybrid_results

In [ ]:
best_mrr = max(result["mrr"] for result in hybrid_results.values())
q6_answer = min(
    k
    for k, result in hybrid_results.items()
    if result["mrr"] == best_mrr
)

print("Q6 best k:", q6_answer)
print("Best MRR:", best_mrr)

## 12. Final answer summary

Run all cells before using these values in the submission form.

In [ ]:
print("Homework 4 answers")
print("1. Average input tokens:", average_input_tokens)
print("2. First text-search result:", q2_answer)
print("3. First vector-search result:", q3_answer)
print("4. Text-search Hit Rate:", q4_value)
print("5. Vector-search MRR:", q5_value)
print("6. Best hybrid-search k:", q6_answer)